[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jalonso1979/puremacro/blob/course-2026A-5.5.0/curso/notebooks/T00_syllabus.ipynb)

> **Google Colab note**: the next cell installs `puremacro` and, if needed, fetches only the `curso/notebooks` folder (helpers, frozen data and models) from the public repository [jalonso1979/puremacro](https://github.com/jalonso1979/puremacro), edition `course-2026A-5.5.0`. It does not ask for Google Drive access.

In [1]:
# Environment setup: Google Colab, local Python or Juno (iPad)
import os
import sys
import subprocess
from pathlib import Path

_course_requirements = ['puremacro==5.5.0']
_course_edition = 'course-2026A-5.5.0'

def _safe_probe(p, is_dir=None):
    """Test paths without raising in the Juno sandbox."""
    try:
        p = Path(p)
        if is_dir is True:
            return p.is_dir()
        if is_dir is False:
            return p.is_file()
        return p.exists()
    except (OSError, PermissionError, TypeError, ValueError):
        return False

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_course_requirements], check=True)
    # A separate directory per edition prevents reuse of helpers from another course.
    _repo = Path("/content/puremacro-" + _course_edition)
    if not _safe_probe(_repo, is_dir=True):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                        "--branch", _course_edition, "https://github.com/jalonso1979/puremacro.git", str(_repo)], check=True)
    subprocess.run(["git", "-C", str(_repo), "checkout", "--detach", _course_edition], check=True)
    subprocess.run(["git", "-C", str(_repo), "sparse-checkout", "set", "curso/notebooks"], check=True)
    _course_status = subprocess.run(["git", "-C", str(_repo), "status", "--porcelain",
                                     "--untracked-files=normal", "--", "curso/notebooks"],
                                    check=True, capture_output=True, text=True).stdout
    if _course_status.strip():
        raise RuntimeError("The cached course edition was modified; use a fresh Colab runtime for the frozen edition.")
    os.chdir(_repo / "curso/notebooks")

# Locate the course helpers for local Python, Colab and Juno.
_candidates = [Path.cwd(), Path.cwd() / "notebooks", Path.cwd() / "curso/notebooks"]
try:
    _candidates.insert(0, Path(__file__).resolve().parent)
except (NameError, OSError):
    pass
for _c in _candidates:
    if _safe_probe(_c / "_nbstyle.py", is_dir=False):
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

import puremacro
if puremacro.__version__ != "5.5.0":
    raise RuntimeError("This course edition requires puremacro==5.5.0; install it and restart the kernel.")

# Cached reference responses are the reproducible default; live models are optional.
os.environ.setdefault("MAV_LLM_MODE", "cache")

%matplotlib inline
try:
    import _nbstyle
    _nbstyle.apply_style()
except ImportError:
    pass

Matplotlib is building the font cache; this may take a moment.


# Advanced Macroeconomics (MAV) — Interactive Syllabus

## An Academic Manifesto: Modern Quantitative Macroeconomics

Modern macroeconomics has moved decisively beyond stylized pencil-and-paper abstractions. The representative-agent Robinson Crusoe economy—while historically vital for establishing intertemporal optimization concepts—cannot address the structural questions facing modern central banks, finance ministries, and multilateral institutions. 

Evaluating the macroeconomic impact of sudden stops, sovereign debt crises, climate policy transitions, or high-frequency monetary surprises requires models that satisfy three core requirements:

1. **Dynamic General Equilibrium Consistency**: Forward-looking households, firms, and policymakers optimize subject to budget and resource constraints, with equilibrium prices clearing goods, labor, and financial asset markets simultaneously.
2. **Microeconomic Heterogeneity and Incomplete Markets**: Wealth inequality, earnings risk, hand-to-mouth households, and credit frictions generate non-linear aggregate dynamics (as captured by Heterogeneous Agent New Keynesian, or HANK, frameworks) that representative-agent models systematically misstate.
3. **Rigorous Empirical Discipline**: Theoretical models are not mere mathematical curiosities; they are structural measurement instruments calibrated or estimated against national accounts, high-frequency financial surprises, and microeconomic survey microdata.

This course, **Advanced Macroeconomics (MAV)**, operationalizes this frontier. Built on **puremacro**—a pure Python computational suite running locally or in cloud Jupyter environments—each topic bridges dynamic mathematical theory, computational algorithms, and real-world empirical estimation.

## Methodological Architecture & Course Dynamics

In this course, models are computational laboratories. Every module is structured around an integrated four-stage research pipeline:

$$\text{Structural Economic Theory} \;\longrightarrow\; \text{Computational Formulation} \;\longrightarrow\; \text{Empirical Pipeline (Micro/Macro Data)} \;\longrightarrow\; \text{Policy Counterfactual Experiments}$$

- **Theory**: Formulate the dynamic optimization problem (Euler equations, Bellman equations, matching functions, or structural VAR identification restrictions).
- **Computational Solution**: Implement algorithms from value function iteration (VFI) to linearized perturbation methods, Kalman filtering, and local projections.
- **Empirical Measurement**: Confront models with real national accounts, FRED series, OECD vintages, and labor survey microdata.
- **Counterfactuals & Stress Tests**: Subject the model to historical stress tests (the 2008 Great Financial Crisis, the 2020 COVID shock, supply-chain dislocations, or sudden stops in emerging markets) to evaluate policy trade-offs.

**Module 1 route:** [A: data](T01_A_data_panels.ipynb) → [B: measurement, vintages and cycles](T01_B_measurement_and_cycles.ipynb) → [C: long-run and short-run facts](T01_C_stylized_facts.ipynb). The diagnostic below is optional and can be revisited after A and B, before the models in Module 2.

**Required text-analysis sections:** in Module 3, [`T03_D`](T03_D_narrative_uncertainty.ipynb), [`T03_E`](T03_E_local_llm_uncertainty.ipynb) and [`T03_F`](T03_F_narrative_llm_central_banks.ipynb), and in Module 9 [`T09_B`](T09_B_build_your_own_index.ipynb), rebuild their text indices from an explicit document-term matrix with `puremacro.text`. `puremacro.text` is included in the course package, `puremacro==5.5.0`. Its document-term matrix section runs in full; a missing import stops execution so the environment can be corrected.

**Module 6 (incomplete markets and HANK)** solves the household problem by value function iteration and by the endogenous grid method, written out by hand. Three sections also declare that dynamic program —states, controls, Bellman equation— in a Dynare-like syntax with `puremacro.dp`: §5 of [`T06_A`](T06_A_wealth_inequality.ipynb) (Aiyagari, Huggett and continuous time), §5 of [`T06_B`](T06_B_life_cycle_and_demographics.ipynb) (the life-cycle hump and a retirement the household chooses) and, in Module 3, §5 of [`T03_A`](T03_A_aggregate_shocks.ipynb) (Krusell–Smith by EGM). **Course edition.** `puremacro.dp` is included in `puremacro==5.5.0`. This section is required: run it with the package installed by the setup cell. A missing import is an environment error; install the pinned package and restart the kernel.

In [2]:
import pandas as pd
from IPython.display import display

modules = pd.DataFrame(
    [
        ("1", "Data, Measurement & Stylized Facts", "A/Q/M panels → vintages and filters → long and short run", "T01_A_data_panels → T01_B_measurement_and_cycles → T01_C_stylized_facts"),
        ("2", "Neoclassical Growth & DSGE Local Solvers", "Dynare .mod, Klein QZ, RBC, New Keynesian, OccBin", "T02_A to T02_F"),
        ("3", "Risk, Uncertainty & AI (LLMs)", "GARCH, GAR, NLP Sentiment", "T00_B, T03_A to T03_F"),
        ("4", "Real Business Cycles (RBC) & Dashboards", "Interactive Simulators", "T04_A, T04_C"),
        ("5", "Empirical Identification (Post-COVID Inflation)", "SVAR, Local Projections", "T05_A to T05_G"),
        ("6", "Incomplete Markets & HANK", "Heterogeneous Agents", "T06_A, T06_B, T06_C"),
        ("7", "Firm Dynamics & Goods Market", "Firm Dynamics & Market Power", "T07_A"),
        ("8", "Labor Flows & Frictions (ENOE Microdata)", "Search & Matching", "T08_A"),
        ("9", "Validation, Frontier & Custom Indexes", "Advanced Econometrics", "T09_A, T09_B"),
    ],
    columns=["Module", "Core Topic", "Tools / Concepts", "Active Labs"],
).set_index("Module")

# Display the curriculum architecture (full cell text, also in plain-text output)
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(modules)

,Core Topic,Tools / Concepts,Active Labs
Module,,,
1,"Data, Measurement & Stylized Facts",A/Q/M panels → vintages and filters → long and short run,T01_A_data_panels → T01_B_measurement_and_cycles → T01_C_stylized_facts
2,Neoclassical Growth & DSGE Local Solvers,"Dynare .mod, Klein QZ, RBC, New Keynesian, OccBin",T02_A to T02_F
3,"Risk, Uncertainty & AI (LLMs)","GARCH, GAR, NLP Sentiment","T00_B, T03_A to T03_F"
4,Real Business Cycles (RBC) & Dashboards,Interactive Simulators,"T04_A, T04_C"
5,Empirical Identification (Post-COVID Inflation),"SVAR, Local Projections",T05_A to T05_G
6,Incomplete Markets & HANK,Heterogeneous Agents,"T06_A, T06_B, T06_C"
7,Firm Dynamics & Goods Market,Firm Dynamics & Market Power,T07_A
8,Labor Flows & Frictions (ENOE Microdata),Search & Matching,T08_A
9,"Validation, Frontier & Custom Indexes",Advanced Econometrics,"T09_A, T09_B"


---

# Macroeconomic Experiment 0: Computational Onboarding Diagnostic Lab

### Economic Motivation: Trends, Non-Stationarity, and Shock Anatomy in Real-Time Data

Before entering the structural models of Module 2, every quantitative macroeconomist must master the computational foundation of empirical macro: **data extraction, transformation, and stationarity diagnostics**.

In aggregate macroeconomic time series, raw national accounts data $Y_t$ (such as Real Gross Domestic Product) exhibit non-stationary stochastic trends with positive average drift:
$$Y_t = Y_0 \exp(\mu t + \xi_t), \quad \xi_t \sim I(1)$$
Standard linear regression and covariance-based spectral filtering (such as the Hodrick-Prescott filter or VAR estimation) require covariance-stationary inputs ($I(0)$). Applying these filters to raw non-stationary levels without understanding their integration order can produce spurious cyclicality and misleading correlation structures (the classic Slutsky-Yule effect).

Furthermore, macroeconomists measure growth rates at differing frequencies:
1. **Quarterly Annualized Growth Rate**:
   $$g_t^q = 400 \times \left( \log Y_t - \log Y_{t-1} \right)$$
   This measures the annualized pace of economic activity during a single quarter, compounding short-run volatility.
2. **Year-over-Year Annual Growth Rate**:
   $$g_t^{yoy} = 100 \times \left( \log Y_t - \log Y_{t-4} \right)$$
   This smooths seasonal idiosyncratic noise and captures medium-term momentum.

In this introductory diagnostic experiment, you will verify your local computational environment, evaluate the historical asymmetry between the **2008 Great Financial Crisis** and the **2020 COVID-19 pandemic shock**, and conduct formal **Augmented Dickey-Fuller (ADF)** stationarity tests to prove why modern macroeconomists must distinguish between $I(1)$ levels and $I(0)$ differences.

### Guided Student Tasks

1. **Pipeline Verification**: Load US quarterly Real GDP by extracting nominal GDP and the GDP deflator from `_datos.gasto()` (or directly from `GDPC1.csv` in `data_curso/`), constructing the chained volume series:
   $$Y_t = \frac{\text{GDP}_t^{\text{nominal}}}{\text{Deflator}_t} \times 100$$
2. **Growth Transformations**: Compute the log level ($100 \log Y_t$), annualized quarterly growth rate ($g_t^q$), and year-over-year rate ($g_t^{yoy}$).
3. **Shock Stress Test**: Isolate the maximum annualized quarterly contraction during the **2008 Great Financial Crisis** (2008Q1–2009Q4) versus the **2020 COVID-19 lockdown shock** (2020Q1–2020Q4). Compare how the annualized quarterly metric amplifies high-frequency shocks relative to year-over-year growth.
4. **Econometric Diagnostic (Unit Root Testing)**: Run the Augmented Dickey-Fuller test (Said & Dickey, 1984) testing the null hypothesis $H_0: \gamma = 0$ (unit root / non-stationarity) in:
   $$\Delta y_t = \alpha + \gamma y_{t-1} + \sum_{j=1}^p \delta_j \Delta y_{t-j} + \varepsilon_t$$
   The specification carries an intercept but no deterministic trend: that is `regression='c'`, the default in `adfuller` and what the code below runs.
   Compare the $p$-value of log Real GDP levels against quarterly growth rates.

In [3]:
import inspect
from types import SimpleNamespace

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
import _datos

# ------------------------------------------------------------------------------
# 1. Pipeline Verification: Load real GDP and compute transformations
# ------------------------------------------------------------------------------
g = _datos.gasto()
pib_usa = g.xs("USA")["gdp"] / g.xs("USA")["gdp_defl"] * 100.0
log_pib = 100.0 * np.log(pib_usa)

# Quarterly annualized growth: 400 * Delta log(Y_t)
g_q = 400.0 * np.log(pib_usa).diff()

# Year-over-year growth: 100 * log(Y_t / Y_{t-4})
g_yoy = 100.0 * np.log(pib_usa / pib_usa.shift(4))

# ------------------------------------------------------------------------------
# 2. Historical Crisis Comparison: 2008 GFC vs. 2020 COVID-19 Shock
# ------------------------------------------------------------------------------
gfc_window = g_q.loc["2008-01-01":"2009-12-31"]
cov_window = g_q.loc["2020-01-01":"2020-12-31"]

gfc_min_q = gfc_window.min()
cov_min_q = cov_window.min()

gfc_trough_date = gfc_window.idxmin().to_period("Q")
cov_trough_date = cov_window.idxmin().to_period("Q")

gfc_min_yoy = g_yoy.loc["2008-01-01":"2009-12-31"].min()
cov_min_yoy = g_yoy.loc["2020-01-01":"2020-12-31"].min()

print("=" * 70)
print("MACROECONOMIC SHOCK ANATOMY: 2008 GFC vs 2020 COVID CONTRACTION")
print("=" * 70)
print(f"2008 GFC Trough (Annualized Quarterly): {gfc_min_q:.2f}% (Trough Quarter: {gfc_trough_date})")
print(f"2008 GFC Trough (Year-over-Year):      {gfc_min_yoy:.2f}%")
print(f"2020 COVID Trough (Annualized Quarterly): {cov_min_q:.2f}% (Trough Quarter: {cov_trough_date})")
print(f"2020 COVID Trough (Year-over-Year):       {cov_min_yoy:.2f}%")
print("-" * 70)

# ------------------------------------------------------------------------------
# 3. Econometric Diagnostic: Augmented Dickey-Fuller Stationarity Test
# ------------------------------------------------------------------------------
# statsmodels 0.15 returns a result object when called with
# ``result_object=True``, and warns about the legacy tuple otherwise; 0.14 --
# still what Colab serves -- does not take that keyword at all and raises
# TypeError.  Read the signature once and give both versions the same two
# attributes, so the notebook opens on either.
_ADF_OBJETO = "result_object" in inspect.signature(adfuller).parameters


def adf(serie):
    """ADF test exposing ``.statistic`` and ``.pvalue`` on statsmodels 0.14 and 0.15."""
    if _ADF_OBJETO:
        return adfuller(serie, result_object=True)
    estadistico, p_valor = adfuller(serie)[:2]
    return SimpleNamespace(statistic=estadistico, pvalue=p_valor)


adf_level = adf(log_pib.dropna())
p_val_level = adf_level.pvalue

adf_diff = adf(g_q.dropna())
p_val_diff = adf_diff.pvalue

print("AUGMENTED DICKEY-FULLER (ADF) TEST RESULTS:")
print(f"Log Real GDP Level:   Test Stat = {adf_level.statistic:.3f}, p-value = {p_val_level:.4f} -> {'Unit Root (Non-Stationary)' if p_val_level > 0.05 else 'Stationary'}")
print(f"Quarterly Growth (g): Test Stat = {adf_diff.statistic:.3f}, p-value = {p_val_diff:.4e} -> {'Unit Root' if p_val_diff > 0.05 else 'Stationary (I(0))'}")
print("=" * 70)

MACROECONOMIC SHOCK ANATOMY: 2008 GFC vs 2020 COVID CONTRACTION
2008 GFC Trough (Annualized Quarterly): -8.85% (Trough Quarter: 2008Q4)
2008 GFC Trough (Year-over-Year):      -4.06%
2020 COVID Trough (Annualized Quarterly): -32.82% (Trough Quarter: 2020Q2)
2020 COVID Trough (Year-over-Year):       -7.69%
----------------------------------------------------------------------
AUGMENTED DICKEY-FULLER (ADF) TEST RESULTS:
Log Real GDP Level:   Test Stat = -1.396, p-value = 0.5841 -> Unit Root (Non-Stationary)
Quarterly Growth (g): Test Stat = -12.935, p-value = 3.6295e-24 -> Stationary (I(0))


In [4]:
# ------------------------------------------------------------------------------
# Verification & Solution Validation
# ------------------------------------------------------------------------------
# Verify shock magnitudes match historical national accounting data
assert gfc_min_q < -8.0, f"Expected GFC annualized contraction < -8%, got {gfc_min_q:.2f}%"
assert cov_min_q < -30.0, f"Expected COVID annualized contraction < -30%, got {cov_min_q:.2f}%"
assert gfc_min_yoy < -3.5, f"Expected GFC YoY contraction < -3.5%, got {gfc_min_yoy:.2f}%"
assert cov_min_yoy < -7.0, f"Expected COVID YoY contraction < -7%, got {cov_min_yoy:.2f}%"

# Verify stationarity diagnostics
assert p_val_level > 0.10, f"Expected log GDP level p-value > 0.10 (unit root), got {p_val_level:.4f}"
assert p_val_diff < 0.01, f"Expected growth rate p-value < 0.01 (covariance stationary), got {p_val_diff:.4e}"

print("✓ Experiment 0 Diagnostic Check Passed: Environment, pipeline, and econometric properties verified successfully!")

✓ Experiment 0 Diagnostic Check Passed: Environment, pipeline, and econometric properties verified successfully!


### Economic Interpretation and Bridge to Module 1

The empirical findings from this onboarding diagnostic establish two indispensable insights for the quantitative macroeconomist:

1. **Shock Profile and Asymmetry**: The 2008 Great Financial Crisis was a prolonged balance-sheet recession triggered by financial sector distress and collateral devaluation; annualized quarterly output fell to $-8.85\%$ at its worst, while the year-over-year rate bottomed at $-4.06\%$. In contrast, the 2020 COVID-19 lockdown was a sudden, violent cessation of economic activity: annualized quarterly growth plunged to $-32.82\%$, an extreme tail shock nearly four times larger than the GFC trough. Yet because the shock was concentrated in 2020Q2, the year-over-year rate fell only to $-7.69\%$.
2. **Integration Order and Econometric Validity**: The ADF test fails to reject the null hypothesis of a unit root for log Real GDP ($p = 0.5841$), confirming that macroeconomic levels are integrated of order 1 ($I(1)$). Attempting to estimate models or evaluate business cycle covariances directly on raw levels violates asymptotic normality and leads to spurious regressions. In contrast, log differences are unambiguously covariance-stationary ($p = 3.63 \times 10^{-24} \ll 0.01$).

If you came here before Module 1, it opens with [`T01_A_data_panels`](T01_A_data_panels.ipynb): it builds the annual, quarterly and monthly panels the rest of the course reads, recording where every number comes from. Then [`T01_B_measurement_and_cycles`](T01_B_measurement_and_cycles.ipynb) asks how vintages and filters shape what we call the cycle. If you are back after A and B, continue with [`T01_C_stylized_facts`](T01_C_stylized_facts.ipynb), which lays out the long-run and short-run facts, and then move on to the models of Module 2. The earlier labs `T01_A_business_cycle_facts` and `T01_B_cross_country_facts` remain as complements.